# Chapter 7 — Understanding the weaknesses

A reproduction shows *that* something does not work; a critique should show *why*. This chapter takes the anomalies of Chapter 6 and traces each to a specific assumption or design decision. Weaknesses are sorted by the kind of evidence behind them:

* **acknowledged** — stated by the authors themselves;
* **observed** — demonstrated here with data from the reproduction;
* **hypothesised** — plausible, not tested.

All demonstrations use the development data only (MS COCO and the development half of the Visual Genome sample), so that the test half stays untouched for Chapter 9.

In [1]:
# Locate the research/ directory and make the real implementation importable.
import sys, json
from pathlib import Path
RESEARCH = Path.cwd().resolve()
while not (RESEARCH / "pel").is_dir():
    RESEARCH = RESEARCH.parent
for path in (RESEARCH, RESEARCH / "experiments"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
FIGURES = RESEARCH / "experiments" / "figures"
RESULTS = RESEARCH / "experiments" / "results"
%matplotlib inline
import numpy as np
import analyse_improvements as analysis           # the real comparison harness (experiments/)
from pel.recorded_access import RecordedAccess
from pel.mapping import map_label, candidate_selection
from pel.disambiguation import disambiguate
from pel.wordnet_access import get_synsets
dev_keys, test_keys = analysis.visual_genome_split()
wikidata = RecordedAccess("coco", "wikidata")

## 7.1 Acknowledged by the authors

| Limitation | Where |
|---|---|
| Labels are linked one at a time; the context of co-perceived entities is unused | Sec. 5.3, 6 |
| Attributes delivered by recognition (colour, size, shape) are unused | Sec. 5.3 |
| Only WordNet alternatives generate candidates | Sec. 5.3 |
| Ordering by links “might not be a good indicator of the candidates' correctness” | Sec. 5.3 |
| Instance-level linking (“Perceived-Entity Recognition”) is not addressed | Sec. 3.3 |
| The baseline is “naive”; a more sophisticated entity-linking approach should be investigated | Sec. 4.2, 5.3 |

## 7.2 Observed weakness 1 — the instance trap

**The design.** Algorithm 2 scores a candidate by the path similarity between its class and the perceived label. Path similarity is symmetric and is largest (1.0) when the two are the same concept.

**The problematic behaviour.** Take `dog` on WikiData:

In [2]:
label = "dog"
pool = candidate_selection(label, wikidata, "L_links")
classes = wikidata.class_labels([c.iri for c in pool])
descriptions = wikidata.descriptions([c.iri for c in pool])
truth = {e["label"]: set(e["ground_truth"]) for e in wikidata.entries}[label]
for rank, (score, c) in enumerate(disambiguate(label, pool, classes)[:6], start=1):
    print(f"{rank}. score {score:.2f}  {'<-- the concept' if c.iri in truth else '':15s} {descriptions[c.iri][:42]:44s} classes: {classes[c.iri][:3]}")
print("...")
ranked = disambiguate(label, pool, classes)
position = next(i for i, (_, c) in enumerate(ranked, start=1) if c.iri in truth)
score, concept = ranked[position - 1]
print(f"{position}. score {score:.2f}  <-- the concept     {descriptions[concept.iri][:42]:44s} classes: {classes[concept.iri][:5]}")

1. score 1.00                  object from Hunterian Art Gallery, Glasgow   classes: ['andiron']
2. score 1.00                  Object from Hunterian Art Gallery, Glasgow   classes: ['andiron']
3. score 0.50                  type of hunting dog used by hunters to tra   classes: ['hunting dog', 'dog type', 'type of hunting dog']
4. score 0.25                  game of chasing other people                 classes: ['game', 'traditional game', 'activity']
5. score 0.25                  sequence of edges connecting a sequence of   classes: ['concept', 'walk']
6. score 0.25                  fictional monster from Dungeons & Dragons    classes: ['fictional dog', 'monster']
...
29. score 0.00  <-- the concept     domesticated mammal of the family Canidae    classes: ['organisms known by a particular common name', 'domesticated mammal', 'animal type']


**The cause.** Look at the two winners: museum objects whose class is *andiron*. WordNet lists “dog” as a synonym of andiron (a firedog), so the class **is** one of the senses of the label and the similarity is the maximum, 1.0. In general, a candidate whose class coincides with a sense of the label is *an instance or a kind* of what the word can denote — one particular andiron, one breed of dog — not the concept itself. The entity that *is* the concept is typed by more general things, so even in the best case it scores 0.5 (class = direct hypernym). Here it scores 0, because its WikiData classes are phrases WordNet does not contain (“domesticated mammal”, “animal type”) — a second problem, taken up as H5 below. The rule “the candidate's class should resemble the label” is satisfied best by the wrong kind of candidate.

This is not a WikiData quirk. YAGO 4.5 contains, for many classes `X`, an entity `X_generic_instance` typed by `X`:

In [3]:
yago = RecordedAccess("coco", "yago")
for label in ("apple", "bicycle", "truck"):
    pool = candidate_selection(label, yago, "L_links")
    classes = yago.class_labels([c.iri for c in pool])
    top = disambiguate(label, pool, classes)[:2]
    print(label, "->", [(round(s, 2), c.iri.split("/")[-1], classes[c.iri][:2]) for s, c in top])

apple -> [(1.0, 'Apple_generic_instance', ['Apple']), (0.08, 'Apple', ['Class', 'fruit of Maloideae'])]
bicycle -> [(1.0, 'Wheel_generic_instance', ['Wheel']), (0.17, 'Cycle_Q5198134', ['Album'])]
truck -> [(1.0, 'Truck_generic_instance', ['Truck']), (0.12, 'Truck', ['Class', 'multi-track vehicle'])]


**How much does it matter?** Count, on the development data, how often $f_c + f_d$ puts a wrong entity first *although the right one is in the pool*, and how often that wrong entity owes its position to a perfect class match (score 1.0):

In [4]:
def instance_trap_statistics(kg):
    rows = []
    for dataset in ("coco", "visual_genome"):
        recording = analysis.load(dataset, kg, "L_links")
        if recording is None:
            return None
        evidence = analysis.Evidence(recording)
        for entry in recording["entries"]:
            if dataset == "visual_genome" and analysis.entry_key(entry) not in dev_keys:
                continue
            table = evidence.get(entry["label"], "all", False)
            truth = set(entry["ground_truth"])
            if not truth & set(table.iris) or not len(table.iris):
                continue
            type_scores = table.matrix[:, 1]
            order = np.argsort(-type_scores, kind="stable")
            if table.iris[order[0]] not in truth:
                rows.append(type_scores[order[0]] == 1.0)
    return len(rows), sum(rows)

for kg in ("wikidata", "dbpedia", "yago"):
    stats = instance_trap_statistics(kg)
    if stats:
        print(f"{kg:9s} ranking errors of f_c+f_d: {stats[0]:3d}   of which the winner has class == label: {stats[1]:3d}  ({stats[1] / max(stats[0], 1):.0%})")

wikidata  ranking errors of f_c+f_d: 144   of which the winner has class == label:  74  (51%)


dbpedia   ranking errors of f_c+f_d:  31   of which the winner has class == label:   0  (0%)


yago      ranking errors of f_c+f_d: 198   of which the winner has class == label: 164  (83%)


**Why it matters.** Perceived-Entity *Typing* — the scenario the baseline addresses — asks for the class. A rule that systematically prefers members of the class over the class is working against the task definition.

**Hypothesis H1.** Replace the symmetric similarity by a *directed* one: reward a candidate when its class is a proper **generalisation** of the label, and give nothing for a class that equals the label.

## 7.3 Observed weakness 2 — senses that cannot be perceived

**The design.** Both algorithms use every WordNet sense of the label.

In [5]:
from pel.candidate_selection import select_candidate_names
for label in ("couch", "bear", "orange"):
    by_pos = {}
    for synset in get_synsets(label):
        by_pos.setdefault(synset.pos(), []).append(synset.name())
    print(label, "senses by part of speech:", {pos: len(v) for pos, v in by_pos.items()})
    print("   names:", [n for _, n in select_candidate_names(label)])

couch senses by part of speech: {'n': 3, 'v': 1}
   names: ['couch', 'lounge', 'sofa', 'cast', 'put', 'redact', 'frame']
bear senses by part of speech: {'n': 2, 'v': 13}
   names: ['bear', 'wear', 'behave', 'deport', 'deliver', 'gestate', 'tolerate', 'pay', 'take over', 'give birth', 'birth', 'stand', 'brook', 'abide', 'carry', 'yield', 'digest', 'endure', 'suffer', 'accept', 'expect', 'stomach', 'support', 'contain', 'comport', 'have a bun in the oven', 'have', 'stick out', 'put up', 'turn out', 'assume', 'hold', 'acquit', 'conduct']
orange senses by part of speech: {'n': 5, 's': 1}
   names: ['orange', 'Orange', 'orangish', 'orangeness', 'orange tree', 'Orange River']


In [6]:
# What the verb senses do to the ranking for "couch":
pool = candidate_selection("couch", wikidata, "L_links")
classes = wikidata.class_labels([c.iri for c in pool])
descriptions = wikidata.descriptions([c.iri for c in pool])
for score, c in disambiguate("couch", pool, classes)[:4]:
    print(f"{score:.2f}  found via {c.source_name!r:8s} {descriptions[c.iri][:60]}")

1.00  found via 'frame'  decorative edging for a picture, such as a painting or photo
0.33  found via 'couch'  piece of furniture for seating two or more persons in the fo
0.33  found via 'couch'  long seating furniture primarily used for reclining rather t
0.20  found via 'frame'  cases or structures that surround or enclose such items as w


“frame” is a synonym of the *verb* “to couch” (to frame a statement); WordNet files both under the synset `frame.v.04`. The picture frame was retrieved through that synonym, and its WikiData class is labelled *frame* — so the class label and the perceived label share a (verb) sense, the similarity is 1.0, and the picture frame overtakes the sofa. The instance trap and the unfiltered senses reinforce each other.

How common is this? Share of labels whose name list contains at least one name that no *noun* sense provides:

In [7]:
for dataset in ("coco", "visual_genome"):
    recording = analysis.load(dataset, "wikidata", "L_links")
    labels = recording["labels"]
    polluted = sum(any(not n["noun"] for n in r["names"]) for r in labels.values())
    non_physical = sum(any(not n["physical"] for n in r["names"]) for r in labels.values())
    print(f"{dataset:14s} labels: {len(labels):4d}   with non-noun names: {polluted / len(labels):.0%}   with names of non-physical senses: {non_physical / len(labels):.0%}")

coco           labels:   80   with non-noun names: 34%   with names of non-physical senses: 41%
visual_genome  labels:  598   with non-noun names: 14%   with names of non-physical senses: 21%


The paper motivates disambiguation with a *physical prior* (“entities that correspond to physical entities should be considerably higher”), but nothing in Algorithms 1–2 encodes it.

**Hypothesis H2.** Restrict the sense inventory to nouns, or to nouns below `physical_entity` in WordNet. (Whether this *helps accuracy* is a separate question — pollution is frequent, but a polluting name only hurts if it retrieves something that then wins. Chapter 9 has the answer.)

## 7.4 Observed weakness 3 — evidence is used one piece at a time

The original pipeline produces a ranking from exactly one signal at a time: outgoing links ($f_q$, $f_c$ with `_links`) **or** class similarity ($f_d$). If the signals were redundant this would cost nothing. They are not:

In [8]:
report = json.loads((RESULTS / "improvements" / "analysis_L_links.json").read_text())
for kg in report["knowledge_graphs"]:
    ranks = report["development"][kg]["ranks"]
    correct = {p: np.array([r == 1 for r in ranks[p]]) for p in ("fq", "fc", "fc+fd")}
    oracle = correct["fq"] | correct["fc"] | correct["fc+fd"]
    print(f"{kg:9s} development hit@1:  f_q {correct['fq'].mean():.3f}   f_c {correct['fc'].mean():.3f}   f_c+f_d {correct['fc+fd'].mean():.3f}"
          f"   | at least one of the three is right: {oracle.mean():.3f}")

wikidata  development hit@1:  f_q 0.618   f_c 0.616   f_c+f_d 0.434   | at least one of the three is right: 0.763
dbpedia   development hit@1:  f_q 0.748   f_c 0.653   f_c+f_d 0.639   | at least one of the three is right: 0.823
yago      development hit@1:  f_q 0.537   f_c 0.592   f_c+f_d 0.158   | at least one of the three is right: 0.650


An oracle that always picked the right process per label would beat every single process by a wide margin. The processes fail on *different* labels, so combining their evidence should pay.

A second symptom: class similarity is zero for most candidates, so the final order is mostly decided by ties.

In [9]:
for kg in report["knowledge_graphs"]:
    recording = analysis.load("coco", kg, "L_links")
    evidence = analysis.Evidence(recording)
    zero = total = 0
    for label in recording["labels"]:
        scores = evidence.get(label, "all", False).matrix[:, 1]
        zero += int((scores == 0).sum()); total += len(scores)
    print(f"{kg:9s} COCO candidates with class similarity exactly 0: {zero / total:.0%} of {total}")

wikidata  COCO candidates with class similarity exactly 0: 42% of 1143
dbpedia   COCO candidates with class similarity exactly 0: 46% of 375


yago      COCO candidates with class similarity exactly 0: 56% of 2415


**Hypothesis H3.** Fuse the evidence: one score that adds type similarity, popularity and name similarity. **Hypothesis H4.** Add evidence the method ignores entirely — the short textual description every graph attaches to its entities. **Hypothesis H5.** Let phrase-like class labels (“fruit of Maloideae”) contribute through their words.

## 7.5 Observed weakness 4 — a recall ceiling, and a noisy yardstick

Chapter 6 showed that for a fifth to a third of the labels the correct entity is never retrieved. Why? For every such Visual Genome entry we fetched the English label of the ground-truth entity and compared it with the names Algorithm 1 tried (`experiments/analyse_retrieval_misses.py`):

In [10]:
misses = json.loads((RESULTS / "additional" / "retrieval_misses.json").read_text())
for kg, data in misses.items():
    print(f"{kg}: {data['not_retrieved']} of {data['entries']} entries never retrieved ({data['not_retrieved'] / data['entries']:.0%})")
    for category, count in data["categories"].items():
        examples = "; ".join(f"{e['label']} → {e['ground_truth_labels'][0] if e['ground_truth_labels'] else '∅'}" for e in data["examples"][category][:3])
        print(f"   {count:4d}  {category:48s} e.g. {examples}")

wikidata: 124 of 600 entries never retrieved (21%)
     10  case differs only                                e.g. alocasia → Alocasia; bacillus → Bacillus; cashew → Anacardium
      1  label has a parenthesised qualifier              e.g. salsa → salsa (food)
     32  name is part of a longer label (or vice versa)   e.g. accessory → fashion accessory; base → military base; Burton → Richard Francis Burton
     77  different word                                   e.g. acetaminophen → paracetamol; airplane propeller → aircraft propeller; alfalfa → hay
      4  ground-truth entity has no English label         e.g. angiosperm → ∅; baguet → ∅; fly → ∅
dbpedia: 32 of 150 entries never retrieved (21%)
      1  case differs only                                e.g. space shuttle → Space Shuttle
      4  label has a parenthesised qualifier              e.g. collar → Collar (clothing); Rhea → Rhea (mythology); Ross → John Ross (Royal Navy officer)
      8  name is part of a longer label (or vice v

Three different things hide behind “not retrieved”:

1. **Fixable by better string matching** — case differences and parenthesised titles (`Cameo (carving)`). A real but bounded gain, largest on YAGO and DBpedia whose labels are Wikipedia page titles.
2. **Label under-specification** — the benchmark label is the first lemma of the synset (“base” for *military base*, “can” for *beverage can*). The label alone does not determine the target; only context could.
3. **Ground truth that is questionable** — the WordNet→WikiData property (P8814) links *accelerator* to a valve-train item and *ape* to the taxon *Hominoidea*. A method that returned a perfectly sensible entity is scored wrong.

Categories 2 and 3 are properties of the **benchmark**, not of the method: they put a ceiling on the accuracy any label-only linker can reach, and they mean absolute accuracies on this benchmark under-state usefulness. Relative comparisons between methods on the same entries remain valid — which is why Chapter 9 uses paired tests.

Related benchmark issues **[ours]**:

* Several Visual Genome entries share a label but have different targets (same word, different synsets); no label-only method can get both right.
* The “all three graphs” filter selects well-documented concepts; the paper's ±2 % margin describes sampling noise, not this selection (Chapter 4.5).
* No interval or significance test accompanies Table 1; with 80 COCO labels, differences below ≈0.15 are within noise.
* The NIL outcome $l_\emptyset$ is part of the task definition but never evaluated.

## 7.6 Observed weakness 5 — fragility to surface conventions *(additional experiment)*

Exact string matching makes the method hostage to label casing. YAGO, queried with the lower-case dataset labels instead of capitalised ones:

In [11]:
lower = json.loads((RESULTS / "additional" / "label_case" / "coco_yago_as_is.json").read_text())
capitalised = json.loads((RESULTS / "reproduction" / "coco_yago.json").read_text())
for condition in ("L|fq", "L_links|fq", "L_links|fc"):
    hits = lambda run: sum(r["first_hit_rank"][condition] == 1 for r in run["records"]) / len(run["records"])
    print(f"COCO → YAGO  {condition:11s} hit@1 with lower-case labels: {hits(lower):.2f}   capitalised: {hits(capitalised):.2f}")

COCO → YAGO  L|fq        hit@1 with lower-case labels: 0.03   capitalised: 0.51
COCO → YAGO  L_links|fq  hit@1 with lower-case labels: 0.03   capitalised: 0.51
COCO → YAGO  L_links|fc  hit@1 with lower-case labels: 0.05   capitalised: 0.55


The paper does not mention casing, although the choice moves accuracy from near zero to above one half on two of its three graphs. Likewise, the order of results for the unordered `S` and `L` queries is decided by the triple store, so their hit@k values are partly an artefact of the database engine.

## 7.7 Hypothesised weaknesses (not tested)

* **Latency.** One label costs about 30 remote queries; a robot cannot link at frame rate without a local copy of the graph or a cache.
* **Detector vocabulary.** With open-vocabulary detectors, labels are free-form phrases; WordNet coverage will drop.
* **Language.** Everything assumes English labels and English graph labels.
* **Dependence on graph modelling.** Algorithm 2 needs class labels that happen to be WordNet entries; graphs whose classes are long descriptive phrases give it nothing to work with.

## 7.8 Priorities

| Weakness | Evidence | Fixable without new data? | Taken up as |
|---|---|---|---|
| instance trap | observed, large | yes | **H1** |
| unperceivable senses | observed, frequent | yes | **H2** |
| evidence used one at a time | observed (oracle gap) | yes | **H3** |
| descriptions ignored | observed (most candidates have no type evidence) | one extra query per label | **H4** |
| phrase-like class labels | observed | yes | **H5** |
| recall ceiling: string matching | observed, bounded | needs new retrieval queries | future work |
| recall ceiling: benchmark | observed | needs a new benchmark | future work |
| NIL, context, attributes | acknowledged / hypothesised | needs images | future work |

**Next:** Chapter 8 turns H1–H5 into code.